In [69]:
import openml
from openml import config
from openml.tasks import get_task
config.apikey = 'c0d6200b271e73a8aec0904980876c3c'
import pandas as pd 
import numpy as np
from typing import List, Optional
import time
from tqdm import tqdm

limit_per_task = 200
delay = 0.05

### Establishing the benchmark suite

In [70]:
suite_num = 225
benchmark_suite = openml.study.get_suite(suite_num)
task_ids = benchmark_suite.tasks

print(f"Benchmark Suite: {benchmark_suite.name}")
print(f"Number of tasks (datasets): {len(task_ids)}")

print(task_ids[:5])
task_id = 16

Benchmark Suite: OpenML100-friendly
Number of tasks (datasets): 54
[6, 11, 12, 14, 16]


### Flows vs Runs
Flows - algorithm + hpo config 
Runs - executions of flow on task

In [71]:
runs_df = openml.runs.list_runs(task=[task_id], output_format='dataframe')
unique_flows = runs_df['flow_id'].unique()

print(f"Found {len(unique_flows)} unique flows used")
print(f"Total runs: {len(runs_df)}")
print(f"Average runs per flow: {len(runs_df) / len(unique_flows):.1f}")

runs_df.head()


Found 748 unique flows used
Total runs: 24209
Average runs per flow: 32.4


,run_id,task_id,setup_id,flow_id,uploader,task_type,upload_time,error_message
0,82,16,2,57,1,TaskType.SUPERVISED_CLASSIFICATION,2014-04-07 00:05:41,
1,91,16,6,61,1,TaskType.SUPERVISED_CLASSIFICATION,2014-04-07 00:10:13,
2,280,16,1,56,1,TaskType.SUPERVISED_CLASSIFICATION,2014-04-07 03:15:18,
3,285,16,4,59,1,TaskType.SUPERVISED_CLASSIFICATION,2014-04-07 03:17:52,
4,324,16,7,62,1,TaskType.SUPERVISED_CLASSIFICATION,2014-04-07 03:48:44,


### predictive accuracy

In [72]:
run_id = 82
run = openml.runs.get_run(run_id)

evaluations = openml.evaluations.list_evaluations(
    function='predictive_accuracy',
    runs=[run_id],
    output_format='dataframe'
)

# Show the accuracy
if not evaluations.empty:
    row = evaluations[evaluations['run_id'] == run_id]
    accuracy = row['value'].values[0]
    print(f"Predictive Accuracy for run {run_id}: {accuracy:.4f}")
else:
    print(f"No accuracy found for run {run_id}")

Predictive Accuracy for run 82: 0.3005


### HPO configurations

In [73]:
setup_id = run.setup_id

# Staep 3: Load the setup
setup = openml.setups.get_setup(setup_id)

# Step 4: Extract hyperparameter configuration
print(f"Hyperparameter configuration for run {run_id}:")
for param in setup.parameters.values():
    print(f"  {param.full_name}: {param.value}")

Hyperparameter configuration for run 82:
  weka.OneR(1)_B: 6


### Mapping flow id to name and parameters

In [74]:
flow = openml.flows.get_flow(62)
print(flow.name)                
print(flow.parameters)          

# all_flows = openml.flows.list_flows(output_format='dataframe', size=None)
# print(f"Total number of flows on OpenML: {len(all_flows)}")

weka.DecisionStump
OrderedDict([('D', None)])


In [75]:
all_runs = []

for task_id in [task_id]: # change
    print(f"\nProcessing task {task_id}...")

    try:
        runs_df = openml.runs.list_runs(task=[task_id], output_format="dataframe")
        if runs_df is None or runs_df.empty:
            print("  No runs found.")
            continue

        if limit_per_task:
            runs_df = runs_df.head(limit_per_task)

        run_ids = runs_df["run_id"].tolist()
        print(f"  Found {len(run_ids)} runs")

        # Get evaluations with error handling
        evals = pd.DataFrame()
        try:
            evals_result = openml.evaluations.list_evaluations(
                function="predictive_accuracy",
                runs=run_ids,
                output_format="dataframe"
            )
            if evals_result is not None:
                evals = evals_result
        except Exception as e:
            print(f"  Failed to get evaluations: {e}")

        for _, row in runs_df.iterrows():
            if row is None:
                continue
                
            run_id = row.get("run_id")
            flow_id = row.get("flow_id")
            
            # Skip if essential IDs are missing
            if run_id is None or flow_id is None:
                print(f"    Skipping row with missing run_id or flow_id")
                continue
                
            accuracy = None

            # Extract accuracy with null checks
            if not evals.empty:
                try:
                    acc = evals[evals["run_id"] == run_id]
                    if not acc.empty and "value" in acc.columns:
                        acc_values = acc["value"].values
                        if len(acc_values) > 0 and acc_values[0] is not None:
                            accuracy = acc_values[0]
                except Exception as e:
                    print(f"    Error extracting accuracy for run {run_id}: {e}")

            hpo_config = {}

            # Get HPO configuration with comprehensive error handling
            try:
                run = openml.runs.get_run(run_id)
                if run is None:
                    print(f"    Run {run_id} returned None")
                elif not hasattr(run, 'setup_id') or run.setup_id is None:
                    print(f"    Run {run_id} has no setup_id")
                else:
                    try:
                        setup = openml.setups.get_setup(run.setup_id)
                        if setup is None:
                            print(f"    Setup {run.setup_id} returned None")
                        elif hasattr(setup, 'parameters') and setup.parameters is not None:
                            for param in setup.parameters.values():
                                if param is not None and hasattr(param, 'full_name') and hasattr(param, 'value'):
                                    if param.full_name is not None:
                                        hpo_config[param.full_name] = param.value
                        else:
                            print(f"    Setup {run.setup_id} has no parameters")
                    except Exception as e:
                        print(f"    Could not load setup {run.setup_id}: {e}")
                        
            except Exception as e:
                print(f"    Could not load HPO config for run {run_id}: {e}")

            # Only append if we have essential data
            if run_id is not None and flow_id is not None:
                all_runs.append({
                    "task_id": task_id,
                    "run_id": run_id,
                    "flow_id": flow_id,
                    "hpo_config": hpo_config,
                    "predictive_accuracy": accuracy
                    
                })
            else:
                print(f"    Skipping run due to missing essential data")

            time.sleep(delay)

    except Exception as e:
        print(f"Error loading task {task_id}: {e}")
        continue  # Continue with next task even if this one fails

# Convert to DataFrame
run_df = pd.DataFrame(all_runs)

print(f"\nTotal collected runs: {len(run_df)}")
run_df.head()

# Save to CSV
csv_filename = f"openml_task_{task_id}_runs.csv"
run_df.to_csv(csv_filename, index=False)

print(f"\nSaved {len(run_df)} runs to '{csv_filename}'")



Processing task 16...
  Found 200 runs
    Setup 7 has no parameters
    Setup 1468 has no parameters
    Setup 1468 has no parameters
    Setup 1839 has no parameters
    Setup 2381 has no parameters
    Setup 2360 has no parameters
    Setup 2372 has no parameters
    Setup 2489 has no parameters
    Setup 2887 has no parameters
    Setup 2891 has no parameters
    Setup 2489 has no parameters
    Setup 2372 has no parameters
    Setup 2899 has no parameters
